# Wrangling big data: when the data won't fit in memory

_Data Wrangling_

**Shrink dtypes, read in chunks, use Parquet, and reach for Polars or Dask before a bigger machine.**

Sooner or later a dataset gets big enough that loading it the naive way &mdash;
       pd.read_csv("everything.csv") &mdash; either crawls or crashes with a memory error.
       The instinct is to ask for a bigger machine. Usually you do not need one. The single most useful
       habit is: don't load what you don't need.

       This lesson is four cheap moves that buy you a lot of room before you reach for heavy machinery:

---

This notebook is a practice scaffold for the **Wrangling big data: when the data won't fit in memory** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q polars
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas + polars

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
from sklearn.datasets import fetch_california_housing

# ============================================================
# 0. PROFILE: where do the bytes actually go?
# ============================================================
d = fetch_california_housing(as_frame=True)
df = d.frame.copy()
df['price_band'] = pd.cut(df['MedHouseVal'], bins=[0, 1, 2, 3, 4, 6],
                          labels=['very_low', 'low', 'mid', 'high', 'very_high']).astype(object)
rng = np.random.RandomState(0)
df['county_code'] = rng.randint(0, 58, size=len(df)).astype('int64')

df.info(memory_usage='deep')                 # see dtypes + per-column memory
print(df.memory_usage(deep=True))            # bytes per column (deep = real string size)
print('before:', df.memory_usage(deep=True).sum())   # -> 2,920,574 bytes (~2.92 MB)

# ============================================================
# 1. MEMORY-EFFICIENT DTYPES: downcast numerics + category text
# ============================================================
for c in df.select_dtypes('float64').columns:
    df[c] = pd.to_numeric(df[c], downcast='float')     # float64 -> float32 (2x)
for c in df.select_dtypes('integer').columns:
    df[c] = pd.to_numeric(df[c], downcast='integer')   # int64 -> int8 here (8x)
df['price_band'] = df['price_band'].astype('category') # object -> category (~60x)

print('after :', df.memory_usage(deep=True).sum())     # -> 784,932 bytes (~0.78 MB) => 3.7x

# ============================================================
# 2. READ IN CHUNKS: aggregate a file bigger than RAM, flat memory
# ============================================================
totals = None
for chunk in pd.read_csv('huge_sales.csv',
                         usecols=['region', 'amount'],   # read only needed columns
                         chunksize=1_000_000):           # 1M rows at a time
    part = chunk.groupby('region')['amount'].sum()
    totals = part if totals is None else totals.add(part, fill_value=0)
print(totals)   # exact global sum-by-region, never holding the whole file

# ============================================================
# 3. PARQUET: columnar format, read only the columns you need
# ============================================================
df.to_parquet('housing.parquet')                          # compressed, columnar
sub = pd.read_parquet('housing.parquet',
                      columns=['MedInc', 'price_band'])    # other columns never read

# ============================================================
# 4. GRADUATE: Polars lazy scan (filter + column choice pushed into the read)
# ============================================================
result = (
    pl.scan_parquet('housing.parquet')      # lazy: nothing read yet
      .filter(pl.col('MedInc') > 5.0)        # predicate pushed down
      .group_by('price_band')
      .agg(pl.col('MedInc').mean().alias('avg_medinc'))
      .collect()                             # only now does it read the needed data
)
print(result)
# For genuinely cluster-scale data, the Dask analog is:
#   import dask.dataframe as dd
#   dd.read_parquet('housing.parquet')[['MedInc','price_band']].groupby('price_band').mean().compute()

## Visualize the data & results

_On a real mixed-dtype DataFrame, how much memory do you save by downcasting numerics (float64->float32, int64->int8) and converting a low-cardinality text column to 'category'?_

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

# Real bundled dataset: 20,640 rows of California housing.
d = fetch_california_housing(as_frame=True)
df = d.frame.copy()

# Add a low-cardinality TEXT column (5 labels) and a small-range INTEGER column,
# both stored with pandas' memory-hungry defaults (object / int64).
df['price_band'] = pd.cut(df['MedHouseVal'], bins=[0, 1, 2, 3, 4, 6],
                          labels=['very_low', 'low', 'mid', 'high', 'very_high']).astype(object)
rng = np.random.RandomState(0)
df['county_code'] = rng.randint(0, 58, size=len(df)).astype('int64')

# ---- BEFORE: group memory by dtype (deep=True counts real string size) ----
mb = df.memory_usage(deep=True)
floats_b = sum(mb[c] for c in df.select_dtypes('float64').columns)
int_b    = mb['county_code']
text_b   = mb['price_band']
total_b  = mb.sum()

# ---- Downcast numerics + convert text to category ----
df2 = df.copy()
for c in df2.select_dtypes('float64').columns:
    df2[c] = pd.to_numeric(df2[c], downcast='float')      # float64 -> float32
df2['county_code'] = pd.to_numeric(df2['county_code'], downcast='integer')  # -> int8
df2['price_band']  = df2['price_band'].astype('category') # object -> category

# ---- AFTER ----
mb2 = df2.memory_usage(deep=True)
floats_a = sum(mb2[c] for c in df2.select_dtypes('float32').columns)
int_a    = mb2['county_code']
text_a   = mb2['price_band']
total_a  = mb2.sum()

for name, b, a in [('floats', floats_b, floats_a), ('int', int_b, int_a),
                   ('text', text_b, text_a), ('TOTAL', total_b, total_a)]:
    print(f'{name:7s} before {b/1e6:6.2f} MB  after {a/1e6:6.2f} MB')
# floats  before   1.49 MB  after   0.74 MB
# int     before   0.17 MB  after   0.02 MB
# text    before   1.27 MB  after   0.02 MB
# TOTAL   before   2.92 MB  after   0.78 MB   (3.7x smaller)
print('reduction:', round(total_b / total_a, 2), 'x')   # -> 3.72 x

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your df.info(memory_usage="deep") shows a 4 GB frame, and one object column named country (only ~200 distinct values across 50 million rows) accounts for most of it. What single change helps most, and roughly why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the column is low-cardinality text: about 200 distinct values, but 50 million rows. — _As object, pandas stores a full Python string per row, so 50M near-duplicate strings dominate the memory._
- Convert it with df["country"] = df["country"].astype("category"). — _category stores each of the ~200 distinct strings once and keeps a tiny integer code per row._
- Re-check memory_usage(deep=True) to confirm the drop. — _With 200 codes the per-row cost falls from a string to about one byte, cutting that column dramatically._

**Answer:** Convert country to the category dtype. Because there are only ~200 distinct values over 50M rows, pandas stores each label once plus a one- or two-byte code per row instead of a whole Python string per row &mdash; a many-x cut on the column that dominates the frame. Downcasting numerics helps too, but the repetitive text column is where the biggest single win is.

</details>

**Problem 2.** A teammate needs the total amount summed by region from a 60 GB CSV on a 16 GB-RAM laptop. pd.read_csv(path) crashes. Outline a chunked approach, and name one operation this approach would not support.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Iterate with for chunk in pd.read_csv(path, usecols=["region","amount"], chunksize=1_000_000):. — _Only two columns and one million rows are in memory at a time, so RAM stays flat regardless of file size._
- Per chunk, compute chunk.groupby("region")["amount"].sum() and add it into a running total Series. — _A grouped sum is associative: partial sums per chunk combine correctly into the global sum._
- After the loop, the accumulated Series is the answer. — _Summing is order-independent, so combining partials gives the exact global result._

**Answer:** Read in chunks with chunksize and usecols=["region","amount"], do a per-chunk groupby(...).sum(), and accumulate the partial sums into a running total. Memory stays at roughly one chunk. What this would not support is any operation that needs all the data at once &mdash; a global sort, an exact median, or a cross-file dedup &mdash; because those cannot be combined from independent per-chunk results. For those, reach for Polars, Dask, or a database.

</details>

**Problem 3.** You repeatedly read 3 of 80 columns from a dataset for different analyses, and each read is slow. The data is currently a single big CSV. What format change helps, and what two mechanisms make it faster?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Convert the CSV to Parquet once: pd.read_csv(path).to_parquet("data.parquet") (chunked if it does not fit). — _Parquet is a compressed columnar format, so it is smaller on disk and laid out by column._
- Read only what you need: pd.read_parquet("data.parquet", columns=["a","b","c"]). — _Columnar storage lets the reader physically skip the bytes of the other 77 columns (column pruning)._
- Add a filter and let the engine skip non-matching row blocks (predicate pushdown), especially via Polars' scan_parquet().filter(). — _Parquet stores per-block statistics, so blocks that cannot match the filter are never read._

**Answer:** Store the data as Parquet instead of CSV. Two mechanisms make repeated partial reads fast: column pruning &mdash; because Parquet is columnar, reading columns=["a","b","c"] physically skips the other 77 columns &mdash; and predicate pushdown &mdash; per-block statistics let the reader skip whole row groups that cannot match a filter. Compression also shrinks the file. A CSV, being row-major text, must scan the whole file every time.

</details>